# PDM advanced: locked files, sessions, full report

Three workflows you'll hit in production:

1. "Who has X checked out?" scans every project for locked files
2. "Where does the open document live in PDM?" cross-references open sessions
3. "Give me a one-screen status report" prints a Program.cs-style showcase

**Prereq:** logged-in PDM session (via Alibre UI), or edit the
connection constants in the first code cell.

## Connect

In [ ]:
from alibrex import connect

# === EDIT IF NEEDED =====================================
PDM_URL      = "http://localhost:8099/"
PDM_DOMAIN   = ""
PDM_USER     = ""
PDM_PASSWORD = ""
# ========================================================

root = connect()
try:
    conn = root.GetActiveServerConnection()
    if conn is None:
        raise RuntimeError("no active session")
except Exception:
    print("  No active connection - connecting directly ...")
    try:
        conn = root.ConnectToPDM(PDM_URL, PDM_DOMAIN, PDM_USER, PDM_PASSWORD)
    except Exception as exc:
        print(f"  ConnectToPDM failed: {exc}")
        raise SystemExit(1)

safe = conn.Safes.Item(0)

## 1. Find every locked file across all projects

Iterative DFS: no recursion, no helper functions.

In [ ]:
from alibrex import IADPDMFolder

stack: list[tuple[str, IADPDMFolder]] = []
for i in range(safe.Projects.Count):
    p = safe.Projects.Item(i)
    stack.append((p.Name, p))

locked = []
checked = 0
while stack:
    path, folder = stack.pop()
    files = folder.FileItems
    for i in range(files.Count):
        fi = files.Item(i)
        checked += 1
        if fi.IsLocked:
            locked.append((path, fi.Name, fi.Extension, fi.LockUser))
    for i in range(folder.Folders.Count):
        sub = folder.Folders.Item(i)
        stack.append((f"{path}/{sub.Name}", sub))

print(f"Scanned {checked} file(s); {len(locked)} locked.")
for path, name, ext, user in locked:
    print(f"  {path}/{name}.{ext}  by {user}")

## 2. Cross-reference open Alibre sessions with PDM

For each open document in the Alibre UI, ask if the file is stored
in a known repository (any safe) and report the PDM reference if so.

In [ ]:
sessions = root.Sessions
print(f"Open sessions: {sessions.Count}")
for i in range(sessions.Count):
    s = sessions.Item(i)
    print(f"\n  [{i}] {s.Name!r}  type={s.SessionType}")
    print(f"        path: {s.FilePath}")
    try:
        from_repo = root.IsOpenedFromRepository(s.FilePath)
    except Exception:
        from_repo = False
    if from_repo:
        ref = root.GetRepositoryReference(s.FilePath)
        print(f"        PDM ref: {ref}")
    else:
        print("        (not in any PDM safe)")

## 3. Full safe summary (one-screen report)

Counts only. For the verbose version, see
`examples/pdm/11_full_pdm_showcase.py`.

In [ ]:
print(f"Safe          : {safe.Name}")
print(f"Connection    : {conn.URL}")
print(f"User          : {conn.UserName}  online={conn.IsOnline}")
print(f"Property defs : {safe.PropertyDefinitions.Count}")
print(f"Classes       : {safe.Classes.Count}")
print(f"Templates     : {safe.Templates.Count}")
print(f"Projects      : {safe.Projects.Count}")
print(f"Libraries     : {safe.Libraries.Count}")
print(f"Recycle bin   : {safe.RecycleBin.Count} item(s)")

## Related material

| | Where |
|---|---|
| 12 progressive `.py` scripts (00-11 + 12_tree) | `examples/pdm/` |
| C# reference port | `Program.cs` (at repo root, source of this curriculum) |
| PDM connection helper | `examples/pdm/_pdm_helper.py` |